# Condition Shift Baseline Notebook

이 노트북은 `condition_shift_baseline` 실험의 thin orchestrator다.

- Colab/서버에서 Git checkout 상태를 확인한다.
- 필요하면 prepared dataset bootstrap script를 호출한다.
- versioned Python runner를 실행한다.
- `summary.json`과 `log.txt`를 읽어 표와 간단 시각화를 보여준다.

실험 핵심 로직은 노트북에 두지 않고, notebook이 호출하는 `.py`는 repo에 versioned 상태로 유지한다.


## 경로 설정

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name in {'ReGraM'}), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

def now_string():
    return datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %Z')

def format_run_label(config):
    return f"{config['shift_family']} / {config['severity']}"

def display_title(title, body=None):
    text = f"## {title}"
    if body:
        text += f"

{body}"
    display(Markdown(text))

def display_run_plan(run_configs):
    rows = []
    for idx, config in enumerate(run_configs, start=1):
        rows.append({
            'order': idx,
            'shift_family': config['shift_family'],
            'severity': config['severity'],
            'manifest_name': config['manifest_name'],
            'summary_name': config['summary_path'].name,
            'wandb': 'on' if config.get('use_wandb') else 'off',
        })
    display(pd.DataFrame(rows))

def display_environment_summary():
    display_title('Environment Summary')
    display(pd.DataFrame([
        {'key': 'REPO_ROOT', 'value': str(REPO_ROOT)},
        {'key': 'EXP_ROOT', 'value': str(EXP_ROOT)},
        {'key': 'REPORT_ROOT', 'value': str(REPORT_ROOT)},
    ]))

run_history = []

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Git Checkout

Colab에서는 먼저 repo를 Git으로 clone 또는 pull 해서 notebook이 사용할 `.py`를 최신 상태로 맞춘다.


In [ ]:
REPO_URL = 'https://github.com/outSeop/ReGraM.git'
REPO_DIR = Path('/content/ReGraM')
git_cmd = (
    f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
    f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
)
print(git_cmd)
subprocess.run(['bash', '-lc', git_cmd], check=True)

REPO_ROOT = REPO_DIR.resolve()
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('updated REPO_ROOT =', REPO_ROOT)
print('updated EXP_ROOT =', EXP_ROOT)


## Dataset load

In [ ]:
from pathlib import Path
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = Path("/content/ReGraM/data/row")
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"

print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

print("done")
print(runtime_root.exists(), runtime_root)


## Optional Dataset Bootstrap

Git checkout 이후 prepared dataset이나 작은 reports 자산이 필요하면 아래처럼 별도 Python bootstrap script를 호출한다.
코드 동기화는 bootstrap이 아니라 Git이 담당한다.


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Runner Orchestration

노트북은 어떤 runner를 어떤 인자로 호출할지 만 관리한다.


## Runner Config

manifest와 category를 먼저 정하고, runner와 summary viewer가 같은 설정을 공유하게 둔다.


In [ ]:
RUNNER_NAME = 'PatchCore manifest shift evaluation'
RUNNER_PATH = EXP_ROOT / 'src' / 'core' / 'run_patchcore_manifest_shift.py'
RUNNER_INPUT_DESC = 'category + manifest jsonl + severity + raw LOCO dataset root'
RUNNER_OUTPUT_DESC = 'summary json + log.txt under reports/patchcore_manifest_shift'

CATEGORY = 'breakfast_box'
MANIFEST_NAMES = [
    'query_motion_blur.jsonl',
    'query_low_light.jsonl',
    'query_gaussian_noise.jsonl',
]
SEVERITIES = ['low', 'medium', 'high']
USE_WANDB = True
WANDB_PROJECT = 'regram-condition-shift'
WANDB_GROUP = 'patchcore-manifest-shift'
WANDB_MODE = 'online'  # online | offline | disabled
WANDB_LOG_IMAGES = True
WANDB_MAX_IMAGES = 2
DEVICE = 'cuda'

run_configs = []
for manifest_name in MANIFEST_NAMES:
    manifest_path = next(
        (root / manifest_name for root in MANIFEST_ROOT_CANDIDATES if (root / manifest_name).exists()),
        None,
    )
    if manifest_path is None:
        searched = [str(root / manifest_name) for root in MANIFEST_ROOT_CANDIDATES]
        raise FileNotFoundError(f'Manifest not found. searched={searched}')

    manifest_stem = manifest_path.stem
    shift_family = manifest_stem.removeprefix('query_') if manifest_stem.startswith('query_') else manifest_stem
    for severity in SEVERITIES:
        summary_path = REPORT_ROOT / 'patchcore_manifest_shift' / f'{CATEGORY}_{shift_family}_{severity}.json'
        runner_cmd = [
            sys.executable,
            str(RUNNER_PATH),
            '--category', CATEGORY,
            '--manifest', str(manifest_path),
            '--severities', severity,
            '--device', DEVICE,
        ]
        if USE_WANDB:
            runner_cmd.extend([
                '--use-wandb',
                '--wandb-project', WANDB_PROJECT,
                '--wandb-group', WANDB_GROUP,
                '--wandb-mode', WANDB_MODE,
            ])
            if WANDB_LOG_IMAGES:
                runner_cmd.extend([
                    '--wandb-log-images',
                    '--wandb-max-images', str(WANDB_MAX_IMAGES),
                ])
        run_configs.append({
            'manifest_name': manifest_name,
            'manifest_path': manifest_path,
            'shift_family': shift_family,
            'severity': severity,
            'summary_path': summary_path,
            'runner_cmd': runner_cmd,
            'use_wandb': USE_WANDB,
        })

display_title('Runner Plan', f'Prepared `{len(run_configs)}` PatchCore runs.')
display(pd.DataFrame([
    {'key': 'runner_name', 'value': RUNNER_NAME},
    {'key': 'runner_path', 'value': str(RUNNER_PATH)},
    {'key': 'runner_inputs', 'value': RUNNER_INPUT_DESC},
    {'key': 'runner_outputs', 'value': RUNNER_OUTPUT_DESC},
    {'key': 'category', 'value': CATEGORY},
    {'key': 'device', 'value': DEVICE},
    {'key': 'wandb_project', 'value': WANDB_PROJECT if USE_WANDB else 'off'},
    {'key': 'wandb_group', 'value': WANDB_GROUP if USE_WANDB else 'off'},
    {'key': 'wandb_mode', 'value': WANDB_MODE if USE_WANDB else 'disabled'},
    {'key': 'wandb_log_images', 'value': WANDB_LOG_IMAGES if USE_WANDB else False},
    {'key': 'wandb_max_images', 'value': WANDB_MAX_IMAGES if USE_WANDB else 0},
]))
display_run_plan(run_configs)


runner_name = PatchCore manifest shift evaluation
runner_path = /content/ReGraM/experiments/validation/condition_shift_baseline/src/core/run_patchcore_manifest_shift.py
runner_inputs = category + manifest jsonl + severity + raw LOCO dataset root
runner_outputs = summary json + log.txt under reports/patchcore_manifest_shift
category = breakfast_box
device = cuda
use_wandb = True


In [36]:
display_title('Run Command Preview', 'Each run executes one `shift_family + severity` pair.')
for config in run_configs:
    print('---')
    print('label =', format_run_label(config))
    print('manifest_path =', config['manifest_path'])
    print('summary_path =', config['summary_path'])
    print('runner_cmd =', ' '.join(config['runner_cmd']))


---
manifest_name = query_motion_blur.jsonl
shift_family = motion_blur
severity = low
manifest_path = /content/ReGraM/manifests/query_motion_blur.jsonl
summary_path = /content/ReGraM/experiments/validation/condition_shift_baseline/reports/patchcore_manifest_shift/breakfast_box_motion_blur_low.json
runner_cmd = /usr/bin/python3 /content/ReGraM/experiments/validation/condition_shift_baseline/src/core/run_patchcore_manifest_shift.py --category breakfast_box --manifest /content/ReGraM/manifests/query_motion_blur.jsonl --severities low --device cuda --use-wandb --wandb-project regram-condition-shift --wandb-group patchcore-manifest-shift --wandb-mode online
---
manifest_name = query_motion_blur.jsonl
shift_family = motion_blur
severity = medium
manifest_path = /content/ReGraM/manifests/query_motion_blur.jsonl
summary_path = /content/ReGraM/experiments/validation/condition_shift_baseline/reports/patchcore_manifest_shift/breakfast_box_motion_blur_medium.json
runner_cmd = /usr/bin/python3 /con

In [ ]:
!mkdir -p /content/ReGraM/external
!test -d /content/ReGraM/external/patchcore-inspection.clean/.git || git clone https://github.com/amazon-science/patchcore-inspection.git /content/ReGraM/external/patchcore-inspection.clean
!pip install faiss-cpu timm


In [33]:
run_history = []
total_runs = len(run_configs)
display_title('Execution Log', f'Started at `{now_string()}` with `{total_runs}` scheduled runs.')

for idx, config in enumerate(run_configs, start=1):
    label = format_run_label(config)
    started_at = now_string()
    print(f"
[{idx}/{total_runs}] START {label} @ {started_at}")
    print('summary_path =', config['summary_path'])
    print('command =', ' '.join(config['runner_cmd']))
    tic = time.perf_counter()
    result = subprocess.run(
        config['runner_cmd'],
        cwd=REPO_ROOT,
        text=True,
    )
    elapsed_sec = round(time.perf_counter() - tic, 2)
    status = 'success' if result.returncode == 0 else 'failed'
    finished_at = now_string()
    run_history.append({
        'order': idx,
        'shift_family': config['shift_family'],
        'severity': config['severity'],
        'status': status,
        'returncode': result.returncode,
        'elapsed_sec': elapsed_sec,
        'started_at': started_at,
        'finished_at': finished_at,
        'summary_path': str(config['summary_path']),
    })
    print(f"[{idx}/{total_runs}] END   {label} -> {status} ({elapsed_sec}s) @ {finished_at}")
    if result.returncode != 0:
        display(pd.DataFrame(run_history))
        raise RuntimeError(f"runner failed for {label}")

display_title('Execution Summary', f'Finished at `{now_string()}`.')
display(pd.DataFrame(run_history))



=== Running: motion_blur / low ===
returncode: 1


RuntimeError: runner failed for motion_blur / low

## Summary Viewer

runner가 남긴 `summary.json`과 `log.txt`를 기준으로만 결과를 본다.


In [ ]:
rows = []
for config in run_configs:
    summary_path = config['summary_path']
    if not summary_path.exists():
        display(Markdown(f"`summary.json` not found: `{summary_path}`"))
        continue

    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    payload = summary.get('payload', {})
    selected_severities = payload.get('selected_severities', [])
    for aug_type, severity_map in payload.get('augmentations', {}).items():
        for severity, item in severity_map.items():
            rows.append({
                'manifest_name': config['manifest_name'],
                'shift_family': config['shift_family'],
                'selected_severity': config['severity'],
                'payload_severities': ','.join(selected_severities),
                'category': summary['class_name'],
                'augmentation_type': aug_type,
                'severity': severity,
                'mean': item.get('mean'),
                'fpr_over_clean_max': item.get('fpr_over_clean_max'),
                'mean_score_shift': item.get('mean_score_shift'),
                'image_auroc_vs_clean_anomaly': item.get('image_auroc_vs_clean_anomaly'),
            })

df = pd.DataFrame(rows).sort_values(['shift_family', 'selected_severity']).reset_index(drop=True)
display_title('Summary Viewer', 'Structured comparison table from `summary.json`.')
display(df)

if run_history:
    display_title('Run History Snapshot')
    display(pd.DataFrame(run_history))
